# Stage 1 / Stage 2 Audit

Audit de transparence sur le retrieval BOFIP clean-room:
1. stage 1 = selection du bon document BOI
2. stage 2 = selection du bon passage dans ce document
3. stage 3 = expansion locale bornee autour du passage

Portee:
- pas de LLM
- pas d'abstention finale
- focus sur les vrais cas reels du benchmark v3


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.jsonio import read_jsonl

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

queries = read_jsonl(PROJECT_ROOT / "data" / "interim" / "retrieval_queries_sample_1000_v3.jsonl")
query_map = {row["id"]: row for row in queries}

doc_eval_base = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_doc_lexical_eval_raw_docs_sample_1000__retrieval_queries_sample_1000_v3.json")
doc_eval_sections = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_doc_lexical_eval_raw_docs_sample_1000__retrieval_queries_sample_1000_v3__sections.json")
doc_eval_sections_firstpara = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_doc_lexical_eval_raw_docs_sample_1000__retrieval_queries_sample_1000_v3__sections_firstpara.json")
local_audit_sections = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_local_strategy_audit_retrieval_queries_sample_1000_v3__sections_body.json")
local_audit_sections_firstpara = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_local_strategy_audit_retrieval_queries_sample_1000_v3__sections_firstpara_body.json")

two_stage_base_chunk = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_retrieval_queries_sample_1000_v3__base_chunk_body.json")
two_stage_sections_chunk = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_retrieval_queries_sample_1000_v3__sections_chunk_body.json")
two_stage_sections_section = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_retrieval_queries_sample_1000_v3__sections_section_then_chunk_body.json")
two_stage_sections_firstpara_chunk = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_two_stage_probe_retrieval_queries_sample_1000_v3__sections_firstpara_chunk_body.json")

deep_dive_sections_chunk = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_deep_dive_probe_retrieval_queries_sample_1000_v3__sections_chunk_body.json")
deep_dive_sections_section = load_json(PROJECT_ROOT / "data" / "reports" / "phase3_deep_dive_probe_retrieval_queries_sample_1000_v3__sections_section_then_chunk_body.json")

doc_base_map = {row["id"]: row for row in doc_eval_base["results"]}
doc_sections_map = {row["id"]: row for row in doc_eval_sections["results"]}
doc_sections_firstpara_map = {row["id"]: row for row in doc_eval_sections_firstpara["results"]}
stage_base_map = {row["id"]: row for row in two_stage_base_chunk["rows"]}
stage_sections_chunk_map = {row["id"]: row for row in two_stage_sections_chunk["rows"]}
stage_sections_section_map = {row["id"]: row for row in two_stage_sections_section["rows"]}
stage_sections_firstpara_chunk_map = {row["id"]: row for row in two_stage_sections_firstpara_chunk["rows"]}
deep_chunk_map = {row["id"]: row for row in deep_dive_sections_chunk["rows"]}
deep_section_map = {row["id"]: row for row in deep_dive_sections_section["rows"]}

print("Loaded v3 stage-comparison artifacts.")


Loaded v3 stage-comparison artifacts.


## 1. Global reading

Ce qu'il faut verifier ici:
- est-ce que `sections` ameliore vraiment le stage 1 documentaire
- est-ce que `section_then_chunk` aide vraiment le stage 2 local
- si non, on ne le garde pas juste parce qu'il est plus sophistique


In [2]:
{
    "doc_eval_base": doc_eval_base["metrics"],
    "doc_eval_sections": doc_eval_sections["metrics"],
    "doc_eval_sections_firstpara": doc_eval_sections_firstpara["metrics"],
    "local_strategy_sections_body": local_audit_sections["summary"],
    "local_strategy_sections_firstpara_body": local_audit_sections_firstpara["summary"],
}


{'doc_eval_base': {'hit@1': 0.9333, 'hit@3': 0.9833, 'hit@5': 0.9833},
 'doc_eval_sections': {'hit@1': 0.95, 'hit@3': 0.9833, 'hit@5': 0.9833},
 'doc_eval_sections_firstpara': {'hit@1': 0.95, 'hit@3': 0.9833, 'hit@5': 1.0},
 'local_strategy_sections_body': {'chunk': {'query_count': 60,
   'doc_hit1_rate': 0.95,
   'top_chunk_evaluated_count': 57,
   'title_only_section_rate': 0.7018,
   'actualite_rate': 0.0,
   'avg_token_count': 235.0,
   'avg_overlap_count': 8.56,
   'avg_overlap_ratio': 0.8078},
  'section_then_chunk': {'query_count': 60,
   'doc_hit1_rate': 0.95,
   'top_chunk_evaluated_count': 57,
   'title_only_section_rate': 0.8596,
   'actualite_rate': 0.0,
   'avg_token_count': 198.7,
   'avg_overlap_count': 7.68,
   'avg_overlap_ratio': 0.7237}},
 'local_strategy_sections_firstpara_body': {'chunk': {'query_count': 60,
   'doc_hit1_rate': 0.95,
   'top_chunk_evaluated_count': 57,
   'title_only_section_rate': 0.6842,
   'actualite_rate': 0.0,
   'avg_token_count': 238.0,
   '

In [3]:
def top_docs(row, limit=3):
    return row["document_hits"][:limit]

def top_sections(row, limit=4):
    return row.get("section_hits", [])[:limit]

def top_chunks(row, limit=4):
    key = "chunk_hits" if "chunk_hits" in row else "top_chunk_hits"
    return row[key][:limit]

def expanded_context(row, limit=6):
    return row["expanded_context"][:limit]

def compare_case(query_id):
    query = query_map[query_id]
    return {
        "id": query_id,
        "query": query["query"],
        "expected_boi": query.get("expected_boi"),
        "expected_behavior": query.get("expected_behavior"),
        "doc_eval_base": {
            "returned_boi": doc_base_map[query_id]["returned_boi"],
            "hit@1": doc_base_map[query_id].get("hit@1"),
            "hit@3": doc_base_map[query_id].get("hit@3"),
        },
        "doc_eval_sections": {
            "returned_boi": doc_sections_map[query_id]["returned_boi"],
            "hit@1": doc_sections_map[query_id].get("hit@1"),
            "hit@3": doc_sections_map[query_id].get("hit@3"),
        },
        "doc_eval_sections_firstpara": {
            "returned_boi": doc_sections_firstpara_map[query_id]["returned_boi"],
            "hit@1": doc_sections_firstpara_map[query_id].get("hit@1"),
            "hit@3": doc_sections_firstpara_map[query_id].get("hit@3"),
        },
        "stage_base_chunk_docs": top_docs(stage_base_map[query_id]),
        "stage_sections_chunk_docs": top_docs(stage_sections_chunk_map[query_id]),
        "stage_sections_firstpara_chunk_docs": top_docs(stage_sections_firstpara_chunk_map[query_id]),
        "stage_sections_chunk_top_chunks": top_chunks(stage_sections_chunk_map[query_id]),
        "stage_sections_firstpara_chunk_top_chunks": top_chunks(stage_sections_firstpara_chunk_map[query_id]),
        "stage_sections_section_top_sections": top_sections(stage_sections_section_map[query_id]),
        "stage_sections_section_top_chunks": top_chunks(stage_sections_section_map[query_id]),
        "deep_dive_sections_chunk": expanded_context(deep_chunk_map[query_id]),
        "deep_dive_sections_section": expanded_context(deep_section_map[query_id]),
    }


## 2. Stage 1 document retrieval: what improved

Cas a lire:
- `q57`: `sections` corrige un vrai parent/child mismatch
- `q32`: `sections_firstpara` transforme un ancien miss `q12` en confusion parent/enfant plus douce
- `q01`: vrai top1 miss toujours ouvert
- `q30`: dernier voisin/famille encore difficile


In [4]:
compare_case("q57")


{'id': 'q57',
 'query': "redevable de la TVA pour les livraisons de biens et prestations de services vue d'ensemble",
 'expected_boi': 'BOI-TVA-DECLA-10-10-20200323',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-TVA-DECLA-10-10-20-20251022',
   'BOI-TVA-DECLA-10-10-20200323',
   'BOI-TVA-DECLA-10-10-30-20-20200902',
   'BOI-TVA-CHAMP-30-30-30-20190109',
   'BOI-TVA-CHAMP-20-50-40-20-20211222'],
  'hit@1': False,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-TVA-DECLA-10-10-20200323',
   'BOI-TVA-DECLA-10-10-20-20251022',
   'BOI-TVA-DECLA-10-10-30-20-20200902',
   'BOI-TVA-CHAMP-30-30-30-20190109',
   'BOI-TVA-CHAMP-20-50-40-20-20211222'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-TVA-DECLA-10-10-20200323',
   'BOI-TVA-DECLA-10-10-20-20251022',
   'BOI-TVA-CHAMP-30-30-30-20190109',
   'BOI-TVA-DECLA-10-10-30-20-20200902',
   'BOI-TVA-DED-10-20-20240724'],
  'hit@1': True,
  'hit@3': True},
 'st

In [5]:
compare_case("q32")


{'id': 'q32',
 'query': 'obligations déclaratives des contribuables pour les plus-values sur biens meubles incorporels',
 'expected_boi': 'BOI-RPPM-PVBMI-40-10-20141014',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-RPPM-PVBMI-40-10-20141014',
   'BOI-RPPM-PVBMI-40-20141014',
   'BOI-RPPM-PVBMI-40-30-70-20230621',
   'BOI-RPPM-PVBMI-40-30-60-10-20230621',
   'BOI-RPPM-PVBMI-10-10-20160304'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-RPPM-PVBMI-40-10-20141014',
   'BOI-RPPM-PVBMI-40-20141014',
   'BOI-RPPM-PVBMI-40-30-60-10-20230621',
   'BOI-RPPM-PVBMI-40-30-60-20-20230621',
   'BOI-RPPM-PVBMI-20-30-40-20191220'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-RPPM-PVBMI-40-20141014',
   'BOI-RPPM-PVBMI-40-10-20141014',
   'BOI-RPPM-PVBMI-40-30-60-10-20230621',
   'BOI-RPPM-PVBMI-40-30-60-20-20230621',
   'BOI-RPPM-PVBMI-20-30-40-20191220'],
  'hit@1': False,
  'hit@3': True},
 

In [6]:
compare_case("q01")


{'id': 'q01',
 'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
 'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-RES-SJ-000014-20210309',
   'BOI-TVA-SECT-80-60-10-50-20120912',
   'BOI-ENR-DG-50-20-20160203',
   'BOI-TVA-DED-50-20-10-20150506',
   'BOI-TVA-DED-50-20-20-20210224'],
  'hit@1': False,
  'hit@3': False},
 'doc_eval_sections': {'returned_boi': ['BOI-BIC-RICI-10-10-50-20250813',
   'BOI-RES-SJ-000014-20210309',
   'BOI-ENR-DG-50-20-20160203',
   'BOI-TVA-DED-50-20-10-20150506',
   'BOI-TVA-DECLA-20-30-10-20-20170301'],
  'hit@1': False,
  'hit@3': False},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BIC-RICI-10-10-50-20250813',
   'BOI-ENR-DG-50-20-20160203',
   'BOI-RES-SJ-000014-20210309',
   'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'BOI-REC-EVTS-10-20-30-10-20150701'],
  'hit@1': False,
  'hit@3': False},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 11.72

In [7]:
compare_case("q30")


{'id': 'q30',
 'query': 'prélèvement forfaitaire obligatoire applicable aux produits de placement à revenu fixe',
 'expected_boi': 'BOI-RPPM-RCM-30-20-40-20220630',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-RPPM-RCM-30-20-70-20191220',
   'BOI-RPPM-RCM-30-20-40-20220630',
   'BOI-RPPM-RCM-30-10-20-40-20140211',
   'BOI-RPPM-RCM-30-10-20-10-20191220',
   'BOI-RPPM-RCM-30-10-20-30-20140211'],
  'hit@1': False,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-RPPM-RCM-30-20-70-20191220',
   'BOI-RPPM-RCM-30-20-40-20220630',
   'BOI-RPPM-RCM-30-10-20-10-20191220',
   'BOI-RPPM-RCM-30-10-20-40-20140211',
   'BOI-RPPM-RCM-30-10-20-30-20140211'],
  'hit@1': False,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-RPPM-RCM-30-20-70-20191220',
   'BOI-RPPM-RCM-30-20-40-20220630',
   'BOI-RPPM-RCM-30-10-20-10-20191220',
   'BOI-RPPM-RCM-30-10-20-40-20140211',
   'BOI-RPPM-RCM-30-10-20-30-20140211'],
  'hit@1': False,
  'hit@3': 

## 3. Stage 2 passage selection: chunk vs section_then_chunk

Ce qu'on cherche:
- `chunk` doit rester passage-centric
- `section_then_chunk` ne vaut la peine que s'il sort un extrait meilleur, pas juste un titre de section ou une introduction generique


In [8]:
compare_case("q03")


{'id': 'q03',
 'query': "crédit d'impôt au titre des avances remboursables sans intérêt pour travaux énergétiques",
 'expected_boi': 'BOI-BIC-RICI-10-110-20-20241127',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-BIC-RICI-10-110-20-20241127',
   'BOI-BIC-RICI-10-115-20-20250409',
   'BOI-BIC-RICI-10-140-20250521',
   'BOI-PAT-IFI-40-20-10-40-20180608',
   'BOI-PAT-IFI-40-20-20-20230503'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-BIC-RICI-10-110-20-20241127',
   'BOI-BIC-RICI-10-115-20-20250409',
   'BOI-BIC-PDSTK-10-10-10-20120912',
   'BOI-BIC-RICI-10-140-20250521',
   'BOI-PAT-ISF-40-40-10-20181011'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BIC-RICI-10-110-20-20241127',
   'BOI-BIC-RICI-10-120-20250917',
   'BOI-BIC-RICI-10-120-20-20250917',
   'BOI-BIC-RICI-10-115-20-20250409',
   'BOI-BIC-PDSTK-10-10-10-20120912'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chun

In [9]:
compare_case("q06")


{'id': 'q06',
 'query': 'convention fiscale France Allemagne traitements publics et pensions',
 'expected_boi': 'BOI-INT-CVB-DEU-10-40-20120912',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-INT-CVB-DEU-10-40-20120912',
   'BOI-INT-CVB-DEU-10-50-20120912',
   'BOI-INT-CVB-DEU-10-70-20141226',
   'BOI-INT-CVB-SGP-20170807',
   'BOI-INT-CVB-MYT-20160608'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-INT-CVB-DEU-10-40-20120912',
   'BOI-INT-CVB-DEU-10-50-20120912',
   'BOI-INT-CVB-DNK-20250521',
   'BOI-INT-CVB-DEU-10-70-20141226',
   'BOI-INT-CVB-DEU-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-INT-CVB-DEU-10-40-20120912',
   'BOI-INT-CVB-DEU-10-50-20120912',
   'BOI-INT-CVB-DEU-10-70-20141226',
   'BOI-INT-CVB-DEU-20120912',
   'BOI-INT-CVB-DNK-20250521'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 35.3256,
   'boi_reference'

In [10]:
compare_case("q14")


{'id': 'q14',
 'query': "compensation et substitution de base légale en contentieux de l'assiette",
 'expected_boi': 'BOI-CTX-DG-20-40-10-20120912',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-CTX-DG-20-40-10-20120912',
   'BOI-CTX-DG-10-10-20120912',
   'BOI-CTX-DG-20-60-20120912',
   'BOI-CTX-PREA-10-50-20120912',
   'BOI-CTX-PREA-10-60-20150204'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-CTX-DG-20-40-10-20120912',
   'BOI-CTX-JUD-30-10-10-20120912',
   'BOI-CTX-JUD-10-40-20-20120912',
   'BOI-CTX-DG-10-10-20120912',
   'BOI-REC-GAR-10-20-20-10-20151007'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-CTX-DG-20-40-10-20120912',
   'BOI-CTX-JUD-30-10-10-20120912',
   'BOI-CTX-JUD-10-40-20-20120912',
   'BOI-CTX-DG-10-10-20120912',
   'BOI-REC-GAR-10-20-20-10-20151007'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 47.3713,
   'boi_ref

In [11]:
compare_case("q31")


{'id': 'q31',
 'query': "plan d'épargne en actions dispositions diverses",
 'expected_boi': 'BOI-RPPM-RCM-40-50-60-20240730',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-RPPM-RCM-40-50-60-20240730',
   'BOI-RPPM-RCM-40-50-20-20240730',
   'BOI-REC-SOLID-10-20-20120912',
   'BOI-RPPM-RCM-40-60-20191220',
   'BOI-REC-SOLID-20-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-RPPM-RCM-40-50-60-20240730',
   'BOI-RPPM-RCM-40-50-20-20240730',
   'BOI-RPPM-RCM-40-60-20191220',
   'BOI-PAT-ISF-30-40-30-20-20181011',
   'BOI-REC-SOLID-10-20-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-RPPM-RCM-40-50-60-20240730',
   'BOI-RPPM-RCM-40-50-20-20240730',
   'BOI-RPPM-RCM-40-60-20191220',
   'BOI-REC-SOLID-20-20120912',
   'BOI-RSA-ES-20-20-20-20170724'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 35.1485,
   'boi_reference': 'BOI-

In [12]:
compare_case("q33")


{'id': 'q33',
 'query': 'plans de règlements accordés par le comptable public suspension des poursuites',
 'expected_boi': 'BOI-REC-PREA-20-10-10-20150506',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-REC-PREA-20-10-10-20150506',
   'BOI-CAD-TOPO-20-10-40-20120912',
   'BOI-TVA-CHAMP-10-20-10-10-20150204',
   'BOI-DJC-EXPC-20-40-20170906',
   'BOI-TVA-DECLA-30-10-20-20-20141117'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-REC-PREA-20-10-10-20150506',
   'BOI-CAD-DIFF-20-20-10-30-20191105',
   'BOI-REC-SOLID-10-20-20120912',
   'BOI-CAD-DIFF-10-20191105',
   'BOI-TVA-CHAMP-40-10-20-20230118'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-REC-PREA-20-10-10-20150506',
   'BOI-BIC-DECLA-30-30-20120912',
   'BOI-CAD-DIFF-20-20-10-30-20191105',
   'BOI-REC-FORCE-30-20191127',
   'BOI-ENR-DG-20120912'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'sc

In [13]:
compare_case("q42")


{'id': 'q42',
 'query': "abattement sur les bénéfices des jeunes agriculteurs période d'application",
 'expected_boi': 'BOI-BA-BASE-30-10-20-20190515',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-BA-BASE-30-10-20-20190515',
   'BOI-BA-BASE-30-10-10-20250723',
   'BOI-RES-IR-000019-20210309',
   'BOI-ENR-DMTOI-10-70-10-20120912',
   'BOI-RES-BIC-000062-20210309'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-BA-BASE-30-10-20-20190515',
   'BOI-BA-BASE-30-10-10-20250723',
   'BOI-RES-IR-000019-20210309',
   'BOI-ENR-DMTOI-10-70-10-20120912',
   'BOI-BIC-CHAMP-80-20-20-20-20240703'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BA-BASE-30-10-20-20190515',
   'BOI-BA-BASE-30-10-10-20250723',
   'BOI-RES-IR-000019-20210309',
   'BOI-ENR-DMTOI-10-70-10-20120912',
   'BOI-BA-BASE-30-20190619'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 48.520

In [14]:
compare_case("q43")


{'id': 'q43',
 'query': "clauses d'indexation des charges financières en BIC",
 'expected_boi': 'BOI-BIC-CHG-50-60-20120912',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-BIC-CHG-50-60-20120912',
   'BOI-BIC-CHG-50-50-10-20190731',
   'BOI-IS-BASE-35-30-20211215',
   'BOI-BIC-CHG-50-10-20120912',
   'BOI-IS-BASE-35-40-10-20-20200513'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-BIC-CHG-50-60-20120912',
   'BOI-IS-BASE-35-40-40-20200513',
   'BOI-IS-BASE-35-30-20211215',
   'BOI-BIC-CHG-50-10-20120912',
   'BOI-BIC-CHG-50-40-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BIC-CHG-50-60-20120912',
   'BOI-BIC-CHG-50-40-20120912',
   'BOI-IS-BASE-35-40-40-20200513',
   'BOI-IS-BASE-35-30-20211215',
   'BOI-BIC-CHG-50-50-40-20120912'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 39.8766,
   'boi_reference': 'BOI-BIC-CHG-50-60-20120

In [15]:
compare_case("q46")


{'id': 'q46',
 'query': "conditions d'éligibilité des jeunes entreprises innovantes et universitaires",
 'expected_boi': 'BOI-BIC-CHAMP-80-20-20-10-20250716',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-BIC-CHAMP-80-20-20-10-20250716',
   'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'BOI-RES-SJ-000014-20210309',
   'BOI-RES-BIC-000154-20241204',
   'BOI-RPPM-RCM-20-10-30-10-20191220'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-BIC-CHAMP-80-20-20-10-20250716',
   'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'BOI-RES-SJ-000014-20210309',
   'BOI-RES-BIC-000154-20241204',
   'BOI-IF-TH-10-40-10-20201222'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BIC-CHAMP-80-20-20-10-20250716',
   'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'BOI-RES-SJ-000014-20210309',
   'BOI-RES-BIC-000154-20241204',
   'BOI-BIC-RICI-10-50-20230208'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'

## 4. Deep-dive local expansion

Le stage 3 n'est pas une boucle infinie. C'est une expansion locale bornee autour des meilleurs chunks.
On verifie ici que cette expansion amene du contexte utile sans noyer le passage top1.


In [16]:
compare_case("q04")


{'id': 'q04',
 'query': "extension du délai de reprise pour l'imposition des revenus de 2018 prélèvement à la source",
 'expected_boi': 'BOI-IR-PAS-50-20-50-20200210',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-IR-PAS-50-20-50-20200210',
   'BOI-CF-PGR-10-70-20161229',
   'BOI-IR-PAS-20-30-30-20180515',
   'BOI-CF-PGR-10-60-20190116',
   'BOI-IR-PAS-10-10-20-20230706'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-IR-PAS-50-20-50-20200210',
   'BOI-IR-PAS-50-10-20-20-20181031',
   'BOI-IR-PAS-50-10-20-30-20200212',
   'BOI-IR-PAS-50-10-30-10-20200210',
   'BOI-IR-PAS-10-10-20-20230706'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-IR-PAS-50-20-50-20200210',
   'BOI-IR-PAS-50-10-30-10-20200210',
   'BOI-IR-PAS-50-10-20-20-20181031',
   'BOI-IR-PAS-20-20-20-20250507',
   'BOI-IR-PAS-50-10-20-30-20200212'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,

In [17]:
compare_case("q36")


{'id': 'q36',
 'query': 'instruction des demandes devant le tribunal administratif en contentieux fiscal',
 'expected_boi': 'BOI-CTX-ADM-10-30-20120912',
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-CTX-ADM-10-30-20120912',
   'BOI-CTX-ADM-10-100-20120912',
   'BOI-CTX-ADM-10-20191030',
   'BOI-CTX-JUD-10-80-20120912',
   'BOI-CTX-JUD-10-50-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections': {'returned_boi': ['BOI-CTX-ADM-10-30-20120912',
   'BOI-CTX-ADM-10-20191030',
   'BOI-CTX-ADM-10-100-20120912',
   'BOI-CTX-PREA-20-20240515',
   'BOI-CTX-JUD-10-80-20120912'],
  'hit@1': True,
  'hit@3': True},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-CTX-ADM-10-30-20120912',
   'BOI-CTX-ADM-10-100-20120912',
   'BOI-CTX-ADM-10-20191030',
   'BOI-CTX-JUD-10-50-20120912',
   'BOI-CTX-PREA-20-20240515'],
  'hit@1': True,
  'hit@3': True},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 46.2326,
   'boi_reference': 'BOI-CTX-ADM-10-30-20120

## 5. Unsupported / false-premise reminders

Pas d'abstention finale encore. Donc ces cas montrent surtout pourquoi il ne faut pas brancher le LLM tout de suite.


In [18]:
compare_case("u04")


{'id': 'u04',
 'query': "l'article 44 sexies-0 A du CGI supprime-t-il le remboursement immédiat du CIR pour les JEI",
 'expected_boi': None,
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-RES-BIC-000062-20210309',
   'BOI-RES-RSA-000127-20250812',
   'BOI-IS-BASE-80-30-20211215',
   'BOI-RES-RPPM-000203-20250724',
   'BOI-BA-RICI-10-20-20210203'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections': {'returned_boi': ['BOI-RES-BIC-000062-20210309',
   'BOI-RES-RSA-000127-20250812',
   'BOI-ENR-DMTG-10-20-30-150-20240627',
   'BOI-BIC-RICI-10-10-50-20250813',
   'BOI-RES-RPPM-000203-20250724'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-BIC-CHAMP-80-20-20-20-20240703',
   'BOI-BIC-RICI-10-50-20230208',
   'BOI-BIC-RICI-10-10-10-20250813',
   'BOI-BIC-CHAMP-80-20-20-10-20250716',
   'BOI-CF-CMSS-60-20230413'],
  'hit@1': None,
  'hit@3': None},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 23.9039,
   'boi_r

In [19]:
compare_case("u05")


{'id': 'u05',
 'query': 'le BOFIP indique-t-il que tous les organismes sans but lucratif sont toujours exonérés de TVA',
 'expected_boi': None,
 'expected_behavior': 'answer',
 'doc_eval_base': {'returned_boi': ['BOI-TVA-DECLA-20-30-20-10-20220323',
   'BOI-IS-CHAMP-10-50-20-20120912',
   'BOI-TVA-CHAMP-10-10-50-30-20120912',
   'BOI-IS-CHAMP-10-50-10-10-20250416',
   'BOI-TVA-LIQ-30-20-90-30-20251022'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections': {'returned_boi': ['BOI-TVA-DECLA-20-30-20-10-20220323',
   'BOI-IS-CHAMP-10-50-10-10-20250416',
   'BOI-TPS-TS-20-20-20190130',
   'BOI-BIC-CHAMP-10-20-20120912',
   'BOI-TCAS-AUT-60-20230330'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-TVA-DECLA-20-30-20-10-20220323',
   'BOI-TVA-CHAMP-30-10-30-20120912',
   'BOI-IS-CHAMP-10-50-10-10-20250416',
   'BOI-TPS-TS-20-20-20190130',
   'BOI-BIC-CHAMP-10-20-20120912'],
  'hit@1': None,
  'hit@3': None},
 'stage_base_chunk_docs': [{'rank': 

In [20]:
compare_case("u10")


{'id': 'u10',
 'query': "procédure de contestation d'un impôt fédéral canadien devant l'administration fiscale française",
 'expected_boi': None,
 'expected_behavior': 'abstain',
 'doc_eval_base': {'returned_boi': ['BOI-CTX-ADM-30-30-20120912',
   'BOI-CTX-ADM-30-60-20120912',
   'BOI-CTX-JUD-10-50-30-20120912',
   'BOI-CTX-JUD-20-20-80-20140626',
   'BOI-CTX-ADM-10-100-20120912'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections': {'returned_boi': ['BOI-CTX-ADM-10-30-20120912',
   'BOI-CTX-ADM-10-100-20120912',
   'BOI-CTX-PREA-10-60-20150204',
   'BOI-CTX-ADM-30-30-20120912',
   'BOI-CTX-JUD-10-50-30-20120912'],
  'hit@1': None,
  'hit@3': None},
 'doc_eval_sections_firstpara': {'returned_boi': ['BOI-CTX-ADM-10-20191030',
   'BOI-CTX-ADM-10-100-20120912',
   'BOI-CTX-PREA-10-50-20120912',
   'BOI-CTX-PREA-10-60-20150204',
   'BOI-CTX-PREA-10-30-20140625'],
  'hit@1': None,
  'hit@3': None},
 'stage_base_chunk_docs': [{'rank': 1,
   'score': 22.5123,
   'boi_reference': 'BOI-CTX

## 6. Engineering conclusion

Lecture stricte attendue:
- stage 1: `sections_firstpara` est le meilleur mode documentaire actuel
- stage 2: `chunk` reste meilleur que `section_then_chunk`
- stage 3: utile comme expansion locale, pas comme substitut au ranking
- prochaine etape: continuer l'audit des derniers misses documentaires et stabiliser la gate d'abstention avant tout LLM
